# OP-07 · Skill brut des modèles : scores, cartes et diagrammes

**Notebook opérationnel** — à exécuter à chaque cycle, après les cumuls C3S et NMME.

| | |
|---|---|
| Étape du workflow | E4 (skill brut, Draft §3.3 et §5) |
| Entrées | valeurs par période des hindcasts (`DATA_OSF/derived/…`) et archives d'observation |
| Sorties | `OUTPUTS_OSF/skill/<cycle>/raw/` : `maps/`, `zones/`, `figures/`, `diagrams/`, `skill_raw_summary.csv` et `OUTPUTS_OSF/registry/models_eligibility.csv` |
| Durée indicative | ~1 h pour 7 modèles C3S × 4 variables + 6 modèles NMME × 2 variables |

Le calcul est **brut** : aucune correction de biais, aucune calibration. Ces scores sont la
référence que la phase P3 devra battre, et le registre d'éligibilité qu'ils produisent décide
quels modèles entrent dans le multimodèle (§3.3).

Trois rappels sur les conventions de la chaîne :

* validation croisée **LOYO** : pour chaque année notée, seuils et climatologies sont recalculés
  sans cette année ;
* les catégories observées viennent des **terciles de l'observation**, les probabilités prévues
  du comptage des membres au-delà des **terciles du modèle lui-même** ;
* **NMME** est livré en moyenne d'ensemble mensuelle : ni décade, ni probabilité brute. Ces
  modèles ne reçoivent que des scores déterministes (voir la section 5).

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
SYSTEMS      = ["c3s", "nmme"]     # NMME : scores déterministes seulement
VARIABLES    = ["precip", "t2m", "tmax", "tmin"]
MODELS       = None                # None = tous les modèles de la configuration
SCALES       = None                # None = décades, mois et saisons (NMME : mois et saisons)
SKIP_R       = False               # True = pas de scores de zone (paquet R verification)

# Figures
SCORES_CARTES = ["pearson", "rpss", "roc_area", "msess"]
DIAG_SCALES   = ["month", "season"]   # diagrammes de fiabilité et ROC
DIAG_ZONES    = ["domain"]            # ajouter "sahel", "equatorial", "austral" si besoin
DIAG_NBOOT    = 300

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
from eccas_s2s.operations.skill_raw import skill_dir

cfg = load_cycle(CYCLE_CONFIG)
OUT = skill_dir(cfg)
print(f"cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")
print(f"sorties : {OUT}")

## 1. Vérifier que R et le paquet `verification` sont disponibles

Les scores de zone et les diagrammes sont calculés en R, comme dans la chaîne de référence du
CAPC-AC. Si cette cellule échoue, installer le paquet (voir `docs/OSF_CHAIN.md`) ou mettre
`SKIP_R = True` pour n'obtenir que les cartes.

In [ ]:
from eccas_s2s.validate.r_bridge import check_packages, RNotAvailable

try:
    print(check_packages())
except RNotAvailable as exc:
    print("R indisponible :", exc)

## 2. Scores et cartes de skill

Écrit une carte par système, modèle et variable (`maps/*_skill.nc`), les scores de zone en R
(`zones/…`), le tableau de synthèse et le registre d'éligibilité.

In [ ]:
from eccas_s2s.operations import skill_raw

ctx = skill_raw.run(CYCLE_CONFIG, systems=SYSTEMS, variables=VARIABLES, models=MODELS,
                    scales=SCALES, skip_r=SKIP_R)
print(f"\nStatut : {ctx.status} — {len(ctx.outputs)} sortie(s)")

Si un run s'interrompt après l'écriture des cartes, le tableau de synthèse se reconstruit
sans tout recalculer :

```python
skill_raw.rebuild_summary(CYCLE_CONFIG)
```

## 3. Tableau de synthèse et éligibilité

In [ ]:
table = pd.read_csv(OUT / "skill_raw_summary.csv")
elig = pd.read_csv(cfg.path_of("output_root") / "registry" / "models_eligibility.csv")
print(f"{len(table)} lignes de score")
display(table[table.scale == "season"]
        .pivot_table(index=["system", "model"], columns="label",
                     values="pearson_domain_median").round(2))
display(elig)

## 4. Cartes de skill

Un panneau par score et par échelle (toutes les périodes côte à côte), plus un panneau
multimodèle par score et période, à la charte CAPC-AC.

In [ ]:
from eccas_s2s.operations import plot_skill_raw

fig_ctx = plot_skill_raw.run(CYCLE_CONFIG, scores=SCORES_CARTES)
print(f"Statut : {fig_ctx.status} — {len(fig_ctx.outputs)} figure(s) dans {OUT / 'figures'}")

## 5. Diagrammes de fiabilité et ROC (R)

Deux figures par période et par zone : la fiabilité (diagramme d'attributs) et la courbe ROC,
les trois catégories sur le même repère. Les couples de **tous les points de grille** de la zone
sont mis en commun, car 24 années seules ne remplissent pas dix classes de probabilité ; les
intervalles à 95 % rééchantillonnent des **années entières**.

Les modèles NMME n'ont pas de probabilité brute : ils sont ignorés ici, et le journal le
signale.

In [ ]:
from eccas_s2s.operations import skill_diagrams

diag = skill_diagrams.run(CYCLE_CONFIG, systems=["c3s"], variables=VARIABLES, models=MODELS,
                          scales=DIAG_SCALES, zones=DIAG_ZONES, n_boot=DIAG_NBOOT)
print(f"Statut : {diag.status} — {len(diag.outputs)} sortie(s) dans {OUT / 'diagrams'}")

## 6. Contrôle visuel rapide

In [ ]:
from IPython.display import Image, display

model, variable, zone = "ecmwf", "precip", "domain"
saison = table[(table.model == model) & (table.variable == variable)
               & (table.scale == "season")].label.iloc[0]
for name in (f"reliability_{saison}.png", f"roc_{saison}.png"):
    f = OUT / "diagrams" / f"c3s_{model}_{variable}" / zone / name
    if f.exists():
        display(Image(filename=str(f), width=680))

## 7. Ce qui est archivé

* `maps/*_skill.nc` — tous les scores par maille, avec le manifeste du run (état git, versions,
  empreintes) dans `OUTPUTS_OSF/runs/<run_id>/`.
* `zones/<modèle>/<zone>/pairs.csv` — les couples eux-mêmes : n'importe quel score peut être
  recalculé ou vérifié plus tard sans relire les hindcasts.
* `skill_raw_summary.csv` et `models_eligibility.csv` — entrées de la pondération multimodèle
  (P6) et du masque de skill des produits (P7).

**Étape suivante :** phase P3, calibration ; les scores calibrés seront comparés à ceux-ci,
période par période et zone par zone.